#Optimisation quantitative de portefeuille.

##Objectif: Développer et comparer plusieurs modèles d'allocation de portefeuille pour optimiser le rendement ajusté au risque, tout en respectant les contraintes de diversification.

##Modèles étudiés:
1. Allocation équipondérée (Equal weight)
2. Allocation inversement proportionelle à la volatilité (Inverse volatility)
3. Minimisation de la variance (Minimum variance)
4. Maximisation du ratio de Sharpe (Maximum Sharpe)
5. Modèle de Black-Litterman

##Contraintes du portefeuille:
1. Capital initial: € 200 000
2. Capital investi: 90%
3. Réserve de liquidités: 10%
4. Nombre minimum de titres: 16
5. Règle de diversification: 5/10/40

##Méthodologie: 

Les modèles seront construits à partir de données financières quotidiennes couvrant les 5 dernières années (une periode commune d'environ 3 ans sera retenue pour comparer équitablement les 5 modèles vu que certains ETF ne disposent pas de 5 années d'historique). Leurs performances seront comparées à l'aide de plusieurs indicateurs : rendement, volatilité, ratio de Sharpe et drawdown maximal.

Les données seront ensuite divisées en deux périodes: une période d'entraînement pour déterminer les poids optimaux du portefeuille et une périod de test hors échantillon pour évaluer les performances des modèles sur des données non utilisées lors de l'optimisation.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [2]:
#parametres et contraintes du portefeuille

initial_capital=200000
investment_fraction= 0.90
cash_fraction= 0.10

min_securities= 16
max_weight= 0.10
large_weight_threshold= 0.05
large_weight_limit= 0.40

invested_capital= initial_capital * investment_fraction
cash_reserve= initial_capital * cash_fraction

print("Capital_initial: €", initial_capital)
print("Capital à investir: €", invested_capital)
print("Réserve de liquidités: €", cash_reserve)

Capital_initial: € 200000
Capital a investir: € 180000.0
Réserve de liquidités: € 20000.0


In [13]:
#Importer les actifs sélectionnés: Les poids du portefeuille seront déterminés indépendamment à partir de nos modèles quantitatifs.

%pip install openpyxl

assets = pd.read_excel(
    "Synapse Capital Market Analysis (1).xlsx",
    sheet_name="Main",
    usecols=["Nom", "Ticker", "Market [Bourse]","Interest"]
)

assets = assets[
    assets["Interest"].fillna("").str.startswith("Interested")
    & (assets["Ticker"] != "3AAP")
].copy()

assets.index = range(1, len(assets) + 1)

print("Nombre d'actifs sélectionnés :", len(assets))

assets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Nombre d'actifs sélectionnés : 22


,Nom,Ticker,Market [Bourse],Interest
1,iS Agri bus Acc,ISAE,Amsterdam,Interested
2,SPDR MSCI W C STAP,WCOS,Amsterdam,Interested
3,LEONARDO,LDO,Milan,Interested
4,SPIE,SPIE,Paris,Interested
5,MSCI Wld Hlth US A,WHCA,Amsterdam,Interested
6,THALES,HO,Paris,Interested
7,ADYEN,ADYEN,Amsterdam,Interested
8,ASML HOLDING,ASML,Amsterdam,Interested
9,TECHNIP ENERGIES,TE,Paris,Interested
10,VANETFMVNUCLXPAR,NUCL,Paris,Interested


In [16]:
#Harmonisation des tickers

exchange_suffix = {
    "Paris": ".PA",
    "Amsterdam": ".AS",
    "Milan": ".MI",
    "Dublin": ".IR"
}

assets["YahooTicker"] = (
    assets["Ticker"].str.strip()
    + assets["Market [Bourse]"].str.strip().map(exchange_suffix)
)

print("Nombre d'actifs :", len(assets))
print("Tickers manquants :", assets["YahooTicker"].isna().sum())

assets[["Nom", "Ticker", "YahooTicker", "Market [Bourse]"]]

Nombre d'actifs : 22
Tickers manquants : 0


,Nom,Ticker,YahooTicker,Market [Bourse]
1,iS Agri bus Acc,ISAE,ISAE.AS,Amsterdam
2,SPDR MSCI W C STAP,WCOS,WCOS.AS,Amsterdam
3,LEONARDO,LDO,LDO.MI,Milan
4,SPIE,SPIE,SPIE.PA,Paris
5,MSCI Wld Hlth US A,WHCA,WHCA.AS,Amsterdam
6,THALES,HO,HO.PA,Paris
7,ADYEN,ADYEN,ADYEN.AS,Amsterdam
8,ASML HOLDING,ASML,ASML.AS,Amsterdam
9,TECHNIP ENERGIES,TE,TE.PA,Paris
10,VANETFMVNUCLXPAR,NUCL,NUCL.PA,Paris


In [15]:
#donnees historique
tickers = assets["YahooTicker"].tolist()
prices_all = yf.download(
    tickers,
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)["Close"]
print("Nombre d'actifs demandés :", len(tickers))
print("Nombre d'actifs téléchargés :", prices_all.shape[1])
print("Nombre de dates :", len(prices_all))
print("Première date :", prices_all.index.min())
print("Dernière date :", prices_all.index.max())

prices_all.head()

Nombre d'actifs demandés : 22
Nombre d'actifs téléchargés : 22
Nombre de dates : 1281
Première date : 2021-10-26 00:00:00
Dernière date : 2026-10-26 00:00:00


Ticker,ADYEN.AS,AM.PA,ASM.AS,ASML.AS,BATT.AS,CRL.MI,ERA.PA,FOHW.PA,GL9.IR,HO.PA,...,LDO.MI,LR.PA,NEX.PA,NUCL.PA,PRY.MI,SPIE.PA,SU.PA,TE.PA,WCOS.AS,WHCA.AS
Date,,,,,,,,,,,,,,,,,,,,,
2021-10-26,2721.5,83.241966,342.702942,647.650696,16.450300,23.434021,68.144913,NaN,12.277943,73.339363,...,6.164671,85.153137,76.397324,NaN,30.365303,18.419727,130.391144,11.770741,37.514999,NaN
2021-10-27,2667.0,82.919502,363.855011,661.843811,16.301600,23.480701,65.435364,NaN,12.207685,73.014778,...,6.106442,85.424896,76.262657,NaN,30.142300,18.507778,133.531357,11.514856,37.435001,NaN
2021-10-28,2630.5,83.380173,373.654877,665.273071,16.196100,23.247295,64.712830,NaN,12.365769,72.600021,...,6.046336,86.312660,76.935966,NaN,30.411758,18.578213,135.923004,11.345712,37.375000,NaN
2021-10-29,2610.5,83.149834,379.767578,667.273499,16.351101,23.340660,65.299904,NaN,12.409682,71.824623,...,5.952419,85.370544,77.788818,NaN,30.383884,18.490166,135.923004,11.558226,37.759998,NaN
2021-11-01,2564.5,83.610504,385.201172,663.558411,16.585501,23.387341,65.525696,NaN,12.339421,72.059036,...,6.042580,86.566315,78.372353,NaN,30.541843,18.754309,136.050797,11.458475,37.775002,NaN


In [19]:
#clean data to find out which dates to use
data_quality = pd.DataFrame({
    "Première date": prices_all.apply(lambda col: col.first_valid_index()),
    "Dernière date": prices_all.apply(lambda col: col.last_valid_index()),
    "Observations": prices_all.notna().sum(),
    "Valeurs manquantes": prices_all.isna().sum()
})

data_quality = data_quality.sort_values("Première date")

data_quality

,Première date,Dernière date,Observations,Valeurs manquantes
Ticker,,,,
ADYEN.AS,2021-10-26,2026-10-26,1281,0
SU.PA,2021-10-26,2026-10-26,1281,0
NEX.PA,2021-10-26,2026-10-26,1281,0
LR.PA,2021-10-26,2026-10-26,1281,0
LDO.MI,2021-10-26,2026-10-26,1271,10
ITWN.AS,2021-10-26,2026-10-26,1279,2
WCOS.AS,2021-10-26,2026-10-26,1279,2
HO.PA,2021-10-26,2026-10-26,1281,0
GL9.IR,2021-10-26,2026-10-26,1272,9


In [20]:
#Recherche d'historiques complémentaires
alternative_prices = yf.download(
    ["NUCL.MI", "FOHW.MI"],
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)["Close"]
#Vérification de la première date disponible
print(alternative_prices.apply(lambda col: col.first_valid_index()))

Ticker
FOHW.MI   2023-07-20
NUCL.MI   2023-04-24
dtype: datetime64[ns]


In [21]:
#construction de bases des donnees
prices_model = prices_all.drop(columns=["NUCL.PA", "FOHW.PA"])
prices_model = prices_model.join(alternative_prices)
prices_3y = prices_model.loc["2023-10-26":"2026-10-26"].copy()
print("Nombre d'actifs :", prices_3y.shape[1])
print("Nombre de dates :", prices_3y.shape[0])
print("\nValeurs manquantes par actif :")
print(prices_3y.isna().sum().sort_values(ascending=False))

Nombre d'actifs : 22
Nombre de dates : 765

Valeurs manquantes par actif :
Ticker
WHCA.AS     66
FOHW.MI      8
NUCL.MI      8
LDO.MI       6
CRL.MI       6
PRY.MI       6
GL9.IR       3
WCOS.AS      2
ITWN.AS      2
ISAE.AS      2
BATT.AS      2
ASML.AS      0
ADYEN.AS     0
AM.PA        0
ASM.AS       0
LR.PA        0
ERA.PA       0
HO.PA        0
NEX.PA       0
TE.PA        0
SPIE.PA      0
SU.PA        0
dtype: int64


In [23]:
#nettoyage des donnees et calculer les rendements

prices_clean = prices_3y.ffill(limit=1).dropna(how="any")

print("Dates initiales :", len(prices_3y))
print("Dates conservées :", len(prices_clean))
print("Dates supprimées :", len(prices_3y) - len(prices_clean))
returns = prices_clean.pct_change(fill_method=None).dropna()

#separaration entrainment et test
train_returns = returns.loc[returns.index < "2025-10-26"]
test_returns = returns.loc[returns.index >= "2025-10-26"]

print("\nObservations d'entraînement :", len(train_returns))
print("Observations de test :", len(test_returns))

print("\nPériode d'entraînement :")
print(train_returns.index.min(), "à", train_returns.index.max())

print("\nPériode de test :")
print(test_returns.index.min(), "à", test_returns.index.max())

Dates initiales : 765
Dates conservées : 701
Dates supprimées : 64

Observations d'entraînement : 509
Observations de test : 191

Période d'entraînement :
2023-10-27 00:00:00 à 2025-10-24 00:00:00

Période de test :
2025-10-27 00:00:00 à 2026-10-26 00:00:00


In [25]:
# Téléchargation WHCA.Londres et le taux de change GBP/EUR, BlackRock confirme que WHCA.L et WHCA.AS correspondent au meme ETF
whca_data = yf.download(
    ["WHCA.L", "GBPEUR=X"],
    period="5y",
    interval="1d",
    auto_adjust=True,
    progress=False
)["Close"]

whca_eur = whca_data["WHCA.L"] * whca_data["GBPEUR=X"]
print("Première date :", whca_eur.first_valid_index())
print("Dernière date :", whca_eur.last_valid_index())

print(
    "Valeurs manquantes depuis janvier 2024 :",
    whca_eur.loc["2024-01-15":].isna().sum()
)

Première date : 2021-11-05 00:00:00
Dernière date : 2026-10-26 00:00:00
Valeurs manquantes depuis janvier 2024 : 17


In [26]:
whca_comparison = pd.DataFrame({
    "Amsterdam_EUR": prices_model["WHCA.AS"],
    "London_EUR": whca_eur.reindex(prices_model.index)
}).loc["2024-01-15":"2026-10-26"]

overlap = whca_comparison.dropna()

ratio = overlap["Amsterdam_EUR"] / overlap["London_EUR"]

print("Dates communes :", len(overlap))
print("Ratio médian des prix :", round(ratio.median(), 4))

print("Valeurs manquantes à Amsterdam :",
      whca_comparison["Amsterdam_EUR"].isna().sum())

print("Valeurs manquantes à Londres :",
      whca_comparison["London_EUR"].isna().sum())

print("Dates manquantes sur les deux marchés :",
      whca_comparison.isna().all(axis=1).sum())

Dates communes : 635
Ratio médian des prix : 1.0002
Valeurs manquantes à Amsterdam : 66
Valeurs manquantes à Londres : 10
Dates manquantes sur les deux marchés : 0


In [27]:
london_eur = whca_eur.reindex(prices_model.index).copy()
london_eur.loc[:"2024-01-14"] = np.nan
prices_model["WHCA.AS"] = prices_model["WHCA.AS"].fillna(london_eur)
prices_3y = prices_model.loc["2023-10-26":"2026-10-26"].copy()
print("Valeurs manquantes pour WHCA :",
      prices_3y["WHCA.AS"].isna().sum())

print("\nValeurs manquantes par actif :")
print(prices_3y.isna().sum().sort_values(ascending=False))

Valeurs manquantes pour WHCA : 0

Valeurs manquantes par actif :
Ticker
FOHW.MI     8
NUCL.MI     8
PRY.MI      6
LDO.MI      6
CRL.MI      6
GL9.IR      3
BATT.AS     2
WCOS.AS     2
ITWN.AS     2
ISAE.AS     2
ASM.AS      0
ASML.AS     0
ADYEN.AS    0
AM.PA       0
NEX.PA      0
LR.PA       0
ERA.PA      0
HO.PA       0
TE.PA       0
SU.PA       0
SPIE.PA     0
WHCA.AS     0
dtype: int64


In [29]:
# Compléter les valeurs manquantes isolées
prices_clean = prices_3y.ffill(limit=1).dropna(how="any")

#rendements quotidiens
returns = prices_clean.pct_change(fill_method=None).dropna()

# Séparation les données d'entraînement et de test
train_returns = returns.loc[returns.index < "2025-10-26"].copy()
test_returns = returns.loc[returns.index >= "2025-10-26"].copy()

print("Nombre d'actifs :", len(returns.columns))
print("Dates initiales :", len(prices_3y))
print("Dates conservées :", len(prices_clean))
print("Dates supprimées :", len(prices_3y) - len(prices_clean))
print("\nObservations d'entraînement :", len(train_returns))
print("Observations de test :", len(test_returns))
print("\nPériode d'entraînement :")
print(train_returns.index.min(), "à", train_returns.index.max())
print("\nPériode de test :")
print(test_returns.index.min(), "à", test_returns.index.max())


Nombre d'actifs : 22
Dates initiales : 765
Dates conservées : 765
Dates supprimées : 0

Observations d'entraînement : 509
Observations de test : 255

Période d'entraînement :
2023-10-27 00:00:00 à 2025-10-24 00:00:00

Période de test :
2025-10-27 00:00:00 à 2026-10-26 00:00:00


Modèle 1: Portefeuille équipondéré
Le capital investi est réparti de manière égale entre les 22 actifs sélectionnés. Ce modèle ne repose sur aucune estimation des rendements futurs et servira de référence pour comparer les autres stratégies d'allocation.

In [30]:
n_assets = train_returns.shape[1]
weights_equal = pd.Series(
    investment_fraction / n_assets,
    index=train_returns.columns
)
allocation_equal = pd.DataFrame({
    "Poids (%)": weights_equal * 100,
    "Montant (€)": weights_equal * initial_capital
})

display(allocation_equal.round(2))

# Vérification des contraintes
print("Nombre d'actifs :", (weights_equal > 0).sum())
print("Poids maximal :", round(weights_equal.max() * 100, 2), "%")
print("Total investi :", round(weights_equal.sum() * 100, 2), "%")
print("Liquidités :", round((1 - weights_equal.sum()) * 100, 2), "%")

,Poids (%),Montant (€)
Ticker,,
ADYEN.AS,4.09,8181.82
AM.PA,4.09,8181.82
ASM.AS,4.09,8181.82
ASML.AS,4.09,8181.82
BATT.AS,4.09,8181.82
CRL.MI,4.09,8181.82
ERA.PA,4.09,8181.82
GL9.IR,4.09,8181.82
HO.PA,4.09,8181.82


Nombre d'actifs : 22
Poids maximal : 4.09 %
Total investi : 90.0 %
Liquidités : 10.0 %
